# 01 --- Hub-and-Spoke Architecture

**CCA Pattern**: The coordinator sits at the hub. Specialized subagents are the spokes. Spokes talk only to the hub, never to each other.

This notebook demonstrates how the coordinator decomposes a research query into subtasks and delegates to specialized agents.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
sys.path.insert(0, str(Path('.').resolve()))

In [ ]:
from research_agents.models.research import SubTask
from research_agents.agent.coordinator import sort_tasks_into_waves
from research_agents.agent.subagents import SUBAGENT_CONFIGS
from research_agents.tools.definitions import ALL_TOOL_SETS

## Understanding the Key Data Structures

### SubTask -- The Unit of Delegation

The `SubTask` model (from `models/research.py`) is the data structure the coordinator uses to delegate work. Each field maps to a CCA concept:

```python
class SubTask(BaseModel):
    task_id: str           # Unique identifier for dependency tracking
    agent_type: str        # Which specialized agent handles this
    instruction: str       # What to do (the "task")
    context: str           # ONLY what the coordinator chose to pass
    depends_on: list[str]  # task_ids that must complete first
```

The critical field is `context`. It contains **only** what the coordinator explicitly selected -- not the coordinator's full conversation history. This is context isolation in data form. We'll explore this deeply in Notebook 02.

### SubagentConfig -- System Prompt + Scoped Tools

Each agent type has a frozen configuration in `agent/subagents.py`:

```python
@dataclass(frozen=True)
class SubagentConfig:
    system_prompt: str   # Agent-specific instructions
    tools: list[dict]    # 4 scoped tools (never the full set)
```

The `frozen=True` means configs are immutable after creation. A web researcher always gets the same 4 tools -- the coordinator cannot accidentally give it fact-checking tools at runtime.

## Correct Pattern: Specialized Subagents

Each agent has its own system prompt and 4 scoped tools.

In [ ]:
# Show each agent's tool count and tool names
for agent_type, config in SUBAGENT_CONFIGS.items():
    tool_names = [t['name'] for t in config.tools]
    print(f'{agent_type}: {len(config.tools)} tools')
    print(f'  Tools: {", ".join(tool_names)}')
    print(f'  Prompt preview: {config.system_prompt[:80].strip()}...')
    print()

### How the Coordinator Delegates

The coordinator's 6-step flow starts with decomposing a query into `SubTask` objects, then topologically sorting them into **waves** based on `depends_on` fields:

- **Wave 0**: Tasks with no dependencies (can run in parallel)
- **Wave 1**: Tasks depending on wave-0 results (sequential)
- **Wave N**: Tasks depending on wave-(N-1) results

The `sort_tasks_into_waves()` function in `coordinator.py` implements this. Let's see it in action:

In [ ]:
# Demonstrate task decomposition into parallel waves
tasks = [
    SubTask(task_id='t1', agent_type='web_researcher',
        instruction='Search for renewable energy data',
        context='Focus on 2024'),
    SubTask(task_id='t2', agent_type='data_extractor',
        instruction='Query capacity statistics',
        context='renewable_capacity table'),
    SubTask(task_id='t3', agent_type='fact_checker',
        instruction='Verify claims',
        context='', depends_on=['t1', 't2']),
]
waves = sort_tasks_into_waves(tasks)
for i, wave in enumerate(waves):
    task_ids = [t.task_id for t in wave]
    print(f'Wave {i}: {task_ids} ({", ".join(t.agent_type for t in wave)})')

Notice that `t1` (web_researcher) and `t2` (data_extractor) are in Wave 0 because they have no dependencies -- they can run in parallel. `t3` (fact_checker) depends on both, so it goes in Wave 1.

### The ServiceContainer -- Dependency Injection

All tool handlers receive a `ServiceContainer` rather than importing services directly:

```python
@dataclass(frozen=True)
class ServiceContainer:
    web_search: WebSearchService
    document_store: DocumentStore
    database: DatabaseService
    knowledge_base: KnowledgeBase
```

The `frozen=True` makes the container immutable. This is dependency injection -- if you later replace `WebSearchService` with a real API client, only the container construction changes. Tool handlers are untouched.

## CCA Exam Tip

> When a CCA exam question asks about the correct architecture for a multi-agent system:
> - **Look for** the option with a single coordinator delegating to specialized subagents
> - **If an answer** describes subagents communicating directly with each other or sharing context automatically, that is the distractor
> - The hub-and-spoke pattern keeps coordination centralized
> - Each spoke (subagent) has its own system prompt, scoped tools, and receives only explicit context